In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import numpy as np
ddir = 'data/'
df = pd.read_csv(os.path.join(ddir,'chembl_augmented_valid.csv'),usecols=['SMILES'])
df.columns = ['smiles']
# df

In [2]:
from rdkit_utils import *

df_med = df[df.smiles.str.len() > 10]
df_med = df_med[df.smiles.str.len() < 20]
df_med['smiles'] = df_med.smiles.apply(lambda x: get_cansmiles(x))
df_med.drop_duplicates(inplace=True)

print(len(df_med))
# df_samp = df_med.sample(1000, replace=False)
# df_samp.head(5)

14002


In [ ]:
from rdkit.Chem import PandasTools

def displaydf(df):
    return HTML(df.to_html(notebook=True))

PandasTools.AddMoleculeColumnToFrame(df_samp,'smiles','mol',includeFingerprints=False)
# displaydf(df_samp[:3])

In [4]:
from collections import Counter
vocab = ''.join(df_med.smiles)
Counter(vocab).keys()

dict_keys(['N', 'C', '(', '=', 'O', ')', 'c', '1', 'l', '#', '[', '-', ']', '+', 'S', 'n', 'F', 'B', 'r', 's', 'P', 'H', 'I', 'o', '2', '3', '4', '5'])

In [ ]:
symb_to_count = {
    'C':61087,
#     'c':45021,
    'N':13227,
#     'n':7120,
    'O':16972,
#     'o':505,
    'F':794,
    'P':334,
    'S':2787,
#     's':974,
    'Cl':1342,
    'Br':580,
    'I':239
}

In [ ]:
def get_weighted_random_atom(symb_to_count):
    tot = sum(symb_to_count.values())
    symb_to_prob = {}
    for k,v in symb_to_count.items():
        symb_to_prob[k] = v/tot 
        
    a = np.array([x for x in symb_to_prob.keys()])
    p = np.array([x for x in symb_to_count.values()])
    p = p / np.sum(p)
    atom_type = np.random.choice(a=a, p=p)
    
    return(atom_type)

In [ ]:
from rdkit import RDLogger  
RDLogger.DisableLog('rdApp.*')

from rdkit_utils import *
from graph_augs import *
import numpy as np

source_to_augs = {}

for _,row in df_samp.iterrows():  
    
    source_to_augs[row['smiles']] = []
    
#     print('- '*40)
    mol = row['mol']
#     mol_show = show_atom_index(mol)
#     display(mol_show)

    idc = [i for i in range(0,(mol.GetNumAtoms()))]
    random.shuffle(idc)
    
    goods = 0 
    for i in idc:
        if goods>10:
            break
        atom_type = weighted_random_atom(symb_to_count)
        try: 
            mol_aug = add_atom_to_mol(mol, atom_type, i)
            if mol_aug.GetNumAtoms()==0:
                continue
            else:
                sm = Chem.MolToSmiles(mol_aug)
#                 print(f"Added {atom_type} to {i}. SMILES: {sm}")
#                 display(mol_aug)
                goods+=1
                source_to_augs[row['smiles']].append(sm)
        except Exception as e:
            continue

In [ ]:
for k,v in source_to_augs.items():
    print(k,len(v))